In [1]:
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import CLIPTextModel, CLIPTokenizer
from torchvision import transforms
from diffusers import UNet2DConditionModel, StableDiffusionPipeline
import os
from accelerate import Accelerator
from PIL import Image

# Configuration parameters
IMAGE_ROOT = "/mnt/Internal/MedImage/unzip_chexpert_images"
OUTPUT_DIR = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/transformer_LORA_guided_latent_fine_tuning_stable_diffusion_model/"
DATA_CSV = "/mnt/Internal/MedImage/chexpert_balanced_for_training_3000_per_label_dis+demog+age.csv"
START_EPOCH = 0
EPOCHS = 15
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
MAX_LENGTH = 77
IMAGE_SIZE = 512
SAVE_EVERY = 1

# Initialize CLIP tokenizer
print("Initializing CLIP tokenizer...")
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

# In the CheXpertData class:
class CheXpertData(torch.utils.data.Dataset):
    def __init__(self, csv_file, image_dir):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),  # Convert grayscale to RGB
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def generate_caption(self, row):
        findings = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
        findings_present = [f for f in findings if row.get(f, 0) == 1]
        findings_str = ", ".join(findings_present).lower() if findings_present else "no significant findings"
        gender = "female" if row.get("GENDER_Female", 0) == 1 else "male"
        age_group = next((col.replace("AGE_GROUP_AGE_", "").replace("_", "-") for col in row.index if "AGE_GROUP_AGE_" in col and row[col] == 1), "unknown")
        race_group = next((col.replace("PRIMARY_RACE_", "").replace("_", " ").lower() for col in row.index if "PRIMARY_RACE_" in col and row[col] == 1), "unknown race")
        return f"X-ray of a {age_group} year-old {race_group} {gender} with {findings_str}."

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = f"{self.image_dir}/{self.data.iloc[idx]['Path']}"
        image = Image.open(img_path).convert("RGB")  # Convert the image to RGB (if not already)
        prompt = self.generate_caption(self.data.iloc[idx])
        encoding = tokenizer(prompt, return_tensors="pt", padding='max_length', max_length=MAX_LENGTH, truncation=True)
        return self.transform(image), encoding['input_ids'].squeeze(0), encoding['attention_mask'].squeeze(0)

### 2. Data Loading
print("Loading CheXpert dataset...")
dataset = CheXpertData(DATA_CSV, IMAGE_ROOT)
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

import torch
import torch.nn as nn
from tqdm import tqdm

class LatentSpaceTransformer(nn.Module):
    def __init__(self, latent_dim, nhead, num_encoder_layers, dim_feedforward):
        super().__init__()
        # Project text embeddings (768) to latent_dim
        self.text_projection = nn.Linear(768, latent_dim)
        
        # Projection from latent channels to latent_dim
        self.latent_projection = nn.Linear(4, latent_dim)
        
        # Transformer architecture
        self.transformer = nn.Transformer(
            d_model=latent_dim, nhead=nhead, 
            num_encoder_layers=num_encoder_layers, 
            dim_feedforward=dim_feedforward
        )
        
        # Convolution layer to reduce channels from latent_dim (768) to 4
        self.conv = nn.Conv2d(latent_dim, 4, kernel_size=1)  # Reducing to 4 channels

    def forward(self, src, tgt):
        batch_size, channels, height, width = src.shape

        # Flatten the latent tensor
        src = src.view(batch_size, -1, channels)  # (batch_size, 4096, 4)

        # Project latents to the target latent_dim
        src = self.latent_projection(src)  # (batch_size, 4096, latent_dim)

        # Project text embeddings to latent_dim and repeat to match the spatial size
        tgt = self.text_projection(tgt)  # (batch_size, seq_length, latent_dim)
        tgt = tgt.mean(dim=1, keepdim=True)  # Aggregate text embedding
        tgt = tgt.repeat(1, src.shape[1], 1)  # (batch_size, 4096, latent_dim)

        # Pass through the transformer
        transformed = self.transformer(src, tgt)

        # Calculate the number of channels in the latent space (latent_dim)
        batch_size, seq_length, latent_dim = transformed.shape
        height, width = 64, 64  # Fixed based on the image size

        # Reshape the transformed output
        transformed = transformed.view(batch_size, latent_dim, height, width)  # (batch_size, latent_dim, 64, 64)

        # Pass through convolution layer to reduce channels to 4
        transformed = self.conv(transformed)  # (batch_size, 4, 64, 64)

        return transformed


### 3. Save and Load Functions
def save_model(model, optimizer, epoch, output_dir):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }
    checkpoint_path = os.path.join(output_dir, f"checkpoint_epoch_{epoch + 1}.pth")
    torch.save(checkpoint, checkpoint_path)
    print(f"Model saved at epoch {epoch + 1}")

def load_model(model, optimizer, checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    print(f"Resuming training from epoch {epoch + 1}")
    return model, optimizer, epoch

from tqdm import tqdm  # Import tqdm for progress bars

# Training Loop
def train():
    accelerator = Accelerator()
    device = accelerator.device

    # Model Setup
    pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float32)
    pipe.to(device)
    unet = pipe.unet

    latent_dim = 768  # CLIP's output embedding dimension
    nhead = 8
    num_encoder_layers = 4
    dim_feedforward = 512

    transformer = LatentSpaceTransformer(latent_dim, nhead, num_encoder_layers, dim_feedforward).to(device)

    # Optimizer
    optimizer = torch.optim.AdamW(list(unet.parameters()) + list(transformer.parameters()), lr=LEARNING_RATE)

    # Load model if checkpoint exists
    checkpoint_path = os.path.join(OUTPUT_DIR, "last_checkpoint.pth")
    if os.path.exists(checkpoint_path):
        unet, optimizer, start_epoch = load_model(unet, optimizer, checkpoint_path)
        transformer, optimizer, _ = load_model(transformer, optimizer, checkpoint_path)
    else:
        start_epoch = 0  # Start from scratch if no checkpoint exists

    for epoch in range(start_epoch, EPOCHS):
        unet.train()
        transformer.train()
        # Wrap data loader with tqdm for progress bar
        for images, input_ids, attention_masks in tqdm(data_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", dynamic_ncols=True):
            images, input_ids, attention_masks = images.to(device), input_ids.to(device), attention_masks.to(device)

            # Encode images into latents using VAE
            latents = pipe.vae.encode(images).latent_dist.sample() * 0.18215

            # Get target: Text embeddings for the transformer model
            encoder_hidden_states = pipe.text_encoder(input_ids).last_hidden_state

            # Pass latents (src) and text embeddings (tgt) through the transformer
            transformed_latents = transformer(latents, encoder_hidden_states)

            # Add noise for diffusion process
            noise = torch.randn_like(transformed_latents)
            timesteps = torch.randint(0, 1000, (transformed_latents.shape[0],), device=device).long()
            noisy_latents = pipe.scheduler.add_noise(transformed_latents, noise, timesteps)

            # Predict the denoised latents
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states).sample

            # Loss calculation using MSE
            loss = nn.functional.mse_loss(model_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1} completed.")

        # Save model checkpoint every `SAVE_EVERY` epochs
        if (epoch + 1) % SAVE_EVERY == 0:
            save_model(unet, optimizer, epoch, OUTPUT_DIR)
            save_model(transformer, optimizer, epoch, OUTPUT_DIR)

if __name__ == "__main__":
    train()


Initializing CLIP tokenizer...
Loading CheXpert dataset...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Epoch 1/15: 100%|██████████| 5946/5946 [4:23:42<00:00,  2.66s/it]  


Epoch 1 completed.
Model saved at epoch 1
Model saved at epoch 1


Epoch 2/15:   9%|▉         | 550/5946 [24:26<3:59:46,  2.67s/it]


KeyboardInterrupt: 